# Семинар 6. AutoML и объяснимость модели (XAI)

**Цель семинара:** Освоить ускоренное прототипирование ML-решений с помощью систем автоматического машинного обучения (AutoML) и научиться вскрывать логику «черного ящика» сложнейших ансамблей методами объяснимого ИИ (SHAP). В корпоративном секторе модель без обоснования прогноза не пройдет комплаенс, поэтому мы научимся генерировать глобальные и локальные интерпретации.

### 🔧 Настройка окружения и импорт библиотек

Для этого семинара вам понадобятся библиотеки `pycaret` (для AutoML) и `shap` (для XAI). Убедитесь, что они установлены в вашем окружении (например, `uv add pycaret shap`).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

try:
    from pycaret.classification import setup, compare_models, pull, finalize_model, save_model
    import shap
except ImportError:
    print("⚠️ Установите пакеты: uv add pycaret shap")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура (Витрина АВТ)

Алгоритмы AutoML обучаются на полностью собранной и очищенной широкой витрине `abt_result.csv`, которую мы подготовили на Семинаре 5.


In [ ]:
INPUT_DIR = os.path.abspath(os.path.join(".", "data", "seminar_5_ml_modeling"))
OUTPUT_DIR = os.path.abspath(os.path.join(".", "data", "seminar_6_automl_shap"))
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

abt_csv_path = os.path.join(INPUT_DIR, "abt_result.csv")

if not os.path.exists(abt_csv_path):
    print(f"⚠️ Витрина не найдена по пути: {abt_csv_path}.")
    print("   Сначала выполните и сохраните результаты Семинара 5!")

df_abt = pd.read_csv(abt_csv_path) if os.path.exists(abt_csv_path) else pd.DataFrame()
if not df_abt.empty:
    print(f"Витрина успешно загружена из Семинара 5. Размерность: {df_abt.shape}")


---

## 🛠 ЗАДАНИЕ 1: Конфигурация AutoML и поиск лучшей модели
**Бизнес-контекст:** Вместо ручного перебора десятков архитектур (Деревья, Регрессии, Бустинги) и их гиперпараметров, бизнес использует AutoML. Это экономит сотни часов Data Science команды. Система сама проводит k-fold кросс-валидацию и возвращает лидерборд алгоритмов.

**Инструкция (TODO):**
1. Инициализируйте пайплайн `setup()` из `pycaret.classification`. Передайте `data=df_abt`, имя вашего таргета, `ignore_features=['Target_ID']`, и `session_id=42` для воспроизводимости.
2. Запустите автоматический поиск лучшей модели с помощью `compare_models()`.
3. Сохраните таблицу результатов кросс-валидации с помощью функции `pull()`.

*🤖 Теги для AI-ментора: `#SEM6_TASK1_START`, `#SEM6_TASK1_BUG`*


In [ ]:
# TODO: 1.1. Вызовите функцию setup(...) передав df_abt, имя таргета и игнорируя Target_ID
# TODO: 1.2. Запустите compare_models() и сохраните в best_model
# TODO: 1.3. Вызовите pull(), чтобы получить DataFrame с метриками и выведите его через display()
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

clf_setup = setup(
    data=..., 
    target=..., 
    ignore_features=[...],
    session_id=42,
    verbose=False
)

best_model = compare_models(...)
leaderboard = ...
display(leaderboard.head(3))


---

## 🛠 ЗАДАНИЕ 2: Глобальный аудит рисков (SHAP Summary Plot)
**Бизнес-контекст:** Топ-менеджменту не нужны метрики `Accuracy`. Им нужно знать, **какие макро-факторы** заставляют клиентов уходить. Основываясь на теории кооперативных игр (векторы Шепли), `SHAP` вычисляет честный вес каждого признака.

**Инструкция (TODO):**
1. Извлеките матрицу признаков, на которой обучалась модель (в PyCaret это `clf_setup.X_train_transformed` или просто подготовьте `X_train` руками).
2. Инициализируйте `shap.TreeExplainer(best_model)` (Ансамбли на основе деревьев отлично работают с `TreeExplainer`).
3. Вычислите SHAP-значения: `shap_values = explainer.shap_values(X_train)`.
4. Постройте график глобальной важности: `shap.summary_plot(shap_values, X_train)`.

*🤖 Теги для AI-ментора: `#SEM6_TASK2_START`, `#SEM6_TASK2_WHY`*


In [ ]:
# TODO: 2.1. Подготовьте X_train (удалите таргет и ID)
# TODO: 2.2. Инициализируйте shap.TreeExplainer, передав best_model
# TODO: 2.3. Рассчитайте explainer.shap_values(X_train)
# TODO: 2.4. Постройте shap.summary_plot
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

X_train = df_abt.drop(columns=[...])

explainer = shap.TreeExplainer(...)
shap_values = explainer.shap_values(...)

# Защита от мультиклассового вывода для деревьев
if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(10, 6))
shap.summary_plot(..., ..., show=False)
plt.show()


---

## 🛠 ЗАДАНИЕ 3: Локальное обоснование ИИ-решения (SHAP Waterfall / Force Plot)
**Бизнес-контекст:** Линейному менеджеру (фронт-офис) предстоит звонить клиенту. Почему именно этому? Алгоритм счел его "в зоне риска". Локальное объяснение покажет, **какие именно триггеры** в истории этого конкретного человека склонили чашу весов.

**Инструкция (TODO):**
1. Выберите одного клиента из базы (например, строку с индексом `idx = 0`).
2. Извлеките его `shap_values` и базовое значение модели (`explainer.expected_value`).
3. Постройте локальный график: `shap.force_plot(...)` (В Jupyter требует `shap.initjs()`) или `shap.waterfall_plot(...)`.


In [ ]:
# TODO: 3.1. Выберите индекс клиента (например, 0)
# TODO: 3.2. Извлеките client_data = X_train.iloc[idx] и client_shap_values = shap_values[idx]
# TODO: 3.3. Визуализируйте локальный вклад
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

idx = ...
client_data = X_train.iloc[...]
client_shap_values = shap_values[...]
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1]

explanation = shap.Explanation(
    values=..., 
    base_values=..., 
    data=..., 
    feature_names=X_train.columns
)
shap.plots.waterfall(explanation)


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция run_automl_and_explain

Для `src/model_training.py` мы напишем функцию-оркестратор `run_automl_and_explain`. Она примет собранную АВТ-витрину, запустит AutoML, оттюнингует лучшую модель, сгенерирует объекты SHAP и вернет кортеж `(model, shap_explainer, metrics)`.


In [ ]:
def run_automl_and_explain(df: pd.DataFrame, target_col: str, task_type: str = 'classification') -> tuple:
    # TODO: Реализуйте setup(), compare_models(), finalize_model()
    # TODO: Инициализируйте shap.TreeExplainer
    # TODO: Сохраните модель через save_model() в директорию MODELS_DIR
    # TODO: Верните кортеж (final_model, explainer, leaderboard)
    raise NotImplementedError("Финальная сборка функции не выполнена!")


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Скрипт проверяет физическое сохранение эталонного артефакта модели `model.pkl` в директорию Семинара 6.


In [ ]:
def run_autocheck(models_dir: str = MODELS_DIR):
    print(f"🚀 Проверка качества и артефактов AutoML в: {models_dir}\n" + "-"*45)
    validation_status = True
    
    model_pkl_path = os.path.join(models_dir, "model.pkl")
    
    # Проверка экспорта
    if os.path.exists(model_pkl_path):
         print(f"✅ Эталонный артефакт model.pkl успешно сериализован в директорию {models_dir}.")
    else:
         print(f"❌ Ошибка: Файл model.pkl не найден по пути '{model_pkl_path}'. Проверьте вызов save_model().")
         validation_status = False

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Пайплайн AutoML & XAI готов.")
        print("Перенесите эту функцию в course_project/src/model_training.py!")
    else:
        print("⚠️ Обнаружены дефекты.")

run_autocheck()